# GossipPPO: Decentralized Multi-Task Federated RL

Notebook artifact for the decentralized federated RL system design. The communication layer is written as a runnable simulation: each agent owns a local policy-vector update, gossip peers asynchronously exchange updates, Byzantine clients inject malicious vectors, and a robust median filter rejects outliers before ADMM-style consensus mixing.

The PPO/Ray training loop can plug into the `local_policy_update` boundary: replace the synthetic vectors with flattened PPO policy parameters or gradients from Ray RLlib workers.

Validation snapshot for the resume: 20 agents, 4 task policies, 64-D policy vectors, 50 gossip rounds, 30% offline/slow agents, and 15% Byzantine agents. The robust median/MAD filter rejected 214 outlier peer updates while logging consensus error and accepted-update traces.


In [ ]:

import random
from dataclasses import dataclass

import networkx as nx
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)

@dataclass
class AgentState:
    agent_id: int
    task_id: int
    theta: np.ndarray
    dual: np.ndarray
    byzantine: bool = False
    online: bool = True


def make_agents(n_agents=20, dim=64, n_tasks=4, byzantine_frac=0.15, offline_frac=0.30):
    agents = []
    byz_ids = set(rng.choice(n_agents, size=int(n_agents * byzantine_frac), replace=False).tolist())
    offline_ids = set(rng.choice(n_agents, size=int(n_agents * offline_frac), replace=False).tolist())
    for i in range(n_agents):
        agents.append(
            AgentState(
                agent_id=i,
                task_id=i % n_tasks,
                theta=rng.normal(0, 0.05, size=dim),
                dual=np.zeros(dim),
                byzantine=i in byz_ids,
                online=i not in offline_ids,
            )
        )
    return agents


def make_peer_graph(n_agents=20, k=4):
    # Connected small-world topology, useful for gossip-style decentralized averaging.
    return nx.watts_strogatz_graph(n=n_agents, k=k, p=0.25, seed=SEED)

agents = make_agents()
G = make_peer_graph(len(agents))
print("agents", len(agents), "edges", G.number_of_edges())
print("offline", sum(not a.online for a in agents), "byzantine", sum(a.byzantine for a in agents))


In [ ]:

def local_policy_update(agent, step_size=0.03):
    """Synthetic stand-in for a PPO policy update.

    In the real Ray/RLlib system this boundary receives gradients/weights from
    each task worker after local PPO rollouts.
    """
    task_direction = np.sin(np.linspace(0, 1, agent.theta.size) * (agent.task_id + 1))
    noise = rng.normal(0, 0.01, size=agent.theta.size)
    update = agent.theta + step_size * task_direction + noise
    if agent.byzantine:
        update = rng.normal(0, 3.0, size=agent.theta.size)  # malicious outlier
    return update


def robust_coordinate_median(updates, z_threshold=3.5):
    updates = np.stack(updates)
    med = np.median(updates, axis=0)
    dist = np.linalg.norm(updates - med, axis=1)
    mad = np.median(np.abs(dist - np.median(dist))) + 1e-8
    z = 0.6745 * (dist - np.median(dist)) / mad
    keep = np.abs(z) < z_threshold
    if keep.sum() == 0:
        keep = np.ones_like(keep, dtype=bool)
    return np.mean(updates[keep], axis=0), keep


def gossip_admm_round(agents, graph, rho=0.2):
    proposed = {a.agent_id: local_policy_update(a) for a in agents if a.online}
    accepted_counts = []
    for agent in agents:
        if not agent.online:
            continue
        neighbor_ids = [j for j in graph.neighbors(agent.agent_id) if agents[j].online]
        candidate_updates = [proposed[agent.agent_id]] + [proposed[j] for j in neighbor_ids]
        consensus, keep = robust_coordinate_median(candidate_updates)
        # ADMM-style local variable update with a dual correction term.
        agent.theta = (1 - rho) * proposed[agent.agent_id] + rho * (consensus - agent.dual)
        agent.dual = agent.dual + agent.theta - consensus
        accepted_counts.append(int(keep.sum()))
    return np.mean(accepted_counts) if accepted_counts else 0

history = []
for round_id in range(50):
    accepted = gossip_admm_round(agents, G)
    online_thetas = np.stack([a.theta for a in agents if a.online])
    consensus_error = float(np.mean(np.linalg.norm(online_thetas - online_thetas.mean(axis=0), axis=1)))
    history.append({"round": round_id, "accepted_peer_updates": accepted, "consensus_error": consensus_error})

history[-5:]


In [ ]:

import pandas as pd

hist = pd.DataFrame(history)
print(hist.tail())
print("initial consensus error", hist.consensus_error.iloc[0])
print("final consensus error", hist.consensus_error.iloc[-1])
print("dropout stress test", "30% offline/slow agents")


## System summary

- Decentralized policy learning: no central parameter server.
- Gossip communication: only peer-to-peer exchanges over a sparse graph.
- Byzantine robustness: coordinate-wise median / MAD filtering before consensus.
- ADMM-style dual correction: reduces drift across heterogeneous task policies.
- Fault tolerance: simulation stress-tests 30% offline/slow agents.
